# 📊 InklusiKerja — Step 5: Evaluasi & Fine-Tuning

**Prerequisite:** Jalankan notebook 01–02 terlebih dahulu agar `data/index/` tersedia.

**Yang dilakukan:**
- Perbandingan model embedding
- Evaluasi clustering disability (Silhouette Score)
- Evaluasi distribusi skor
- Panduan fine-tuning dengan Contrastive Learning

## 1. Perbandingan Model

In [ ]:
import numpy as np
import pandas as pd
import json
import pickle

comparison = pd.DataFrame([
    {"Model": "paraphrase-multilingual-MiniLM-L12-v2", "Ukuran": "~450MB", "Dim": 384,
     "Bahasa ID": "⭐⭐⭐", "Speed": "⭐⭐⭐⭐⭐", "Akurasi": "⭐⭐⭐", "Use Case": "Prototyping"},
    {"Model": "paraphrase-multilingual-mpnet-base-v2", "Ukuran": "~1.1GB", "Dim": 768,
     "Bahasa ID": "⭐⭐⭐⭐", "Speed": "⭐⭐⭐⭐", "Akurasi": "⭐⭐⭐⭐", "Use Case": "Produksi (general)"},
    {"Model": "LazarusNLP/indobert-base-p2", "Ukuran": "~500MB", "Dim": 768,
     "Bahasa ID": "⭐⭐⭐⭐⭐", "Speed": "⭐⭐⭐", "Akurasi": "⭐⭐⭐⭐⭐", "Use Case": "✅ TERBAIK untuk InklusiKerja"},
    {"Model": "LazarusNLP/IndoNanoT5-base", "Ukuran": "~850MB", "Dim": 768,
     "Bahasa ID": "⭐⭐⭐⭐⭐", "Speed": "⭐⭐", "Akurasi": "⭐⭐⭐⭐", "Use Case": "Alternatif"},
])

print("=" * 70)
print("PERBANDINGAN MODEL EMBEDDING")
print("=" * 70)
print(comparison.to_string(index=False))
print("\n💡 Rekomendasi:")
print("   Hackathon : paraphrase-multilingual-MiniLM-L12-v2 (cepat setup)")
print("   Produksi  : LazarusNLP/indobert-base-p2 (akurasi tertinggi untuk ID)")

## 2. Load Evaluator

In [ ]:
EMBEDDINGS_PATH = "data/index/job_embeddings.npy"
METADATA_PATH   = "data/index/jobs_metadata.pkl"

embeddings = np.load(EMBEDDINGS_PATH)
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

print(f"✅ Evaluator dimuat:")
print(f"   Embeddings : {embeddings.shape}")
print(f"   Metadata   : {len(metadata)} entri")

## 3. Evaluasi — Disability Clustering (Silhouette Score)

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder

labels = [m["disability_type"] for m in metadata.values()]
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)

sil_score = silhouette_score(
    embeddings, encoded_labels,
    metric="cosine",
    sample_size=min(500, len(embeddings)),
)

print(f"\n📊 Disability Clustering Evaluation")
print(f"   Silhouette Score : {sil_score:.4f}")
print(f"   Jumlah Kelas     : {len(le.classes_)}")
print(f"   Kelas            : {list(le.classes_)}")

if sil_score > 0.5:
    grade = "A — Model bekerja sangat baik"
elif sil_score > 0.3:
    grade = "B — Model bekerja baik, ada ruang improvement"
elif sil_score > 0.1:
    grade = "C — Perlu fine-tuning atau data lebih banyak"
else:
    grade = "D — Fine-tuning sangat direkomendasikan"

print(f"\n🏆 Overall Grade: {grade}")

verdict = '✅ Clustering bagus' if sil_score > 0.3 else '⚠️ Clustering lemah — pertimbangkan fine-tuning'
print(f"   {verdict}")

## 4. Evaluasi — Distribusi Skor

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Sample cosine similarity antar random pair
np.random.seed(42)
n_sample = min(200, len(embeddings))
idx_a = np.random.choice(len(embeddings), n_sample, replace=False)
idx_b = np.random.choice(len(embeddings), n_sample, replace=False)
sample_scores = [float(np.dot(embeddings[a], embeddings[b])) * 100 for a, b in zip(idx_a, idx_b)]

stats = {
    "mean": float(np.mean(sample_scores)),
    "std" : float(np.std(sample_scores)),
    "min" : float(np.min(sample_scores)),
    "max" : float(np.max(sample_scores)),
    "p25" : float(np.percentile(sample_scores, 25)),
    "p75" : float(np.percentile(sample_scores, 75)),
}

print(f"\n📊 Score Distribution (cosine similarity antar random pair)")
for k, v in stats.items():
    print(f"   {k:5s}: {v:.2f}")

if stats["std"] < 5:
    print("   ⚠️ Distribusi terlalu sempit — model kurang diskriminatif")
elif stats["std"] > 30:
    print("   ⚠️ Distribusi terlalu lebar — cek normalisasi")
else:
    print("   ✅ Distribusi skor sehat")

In [ ]:
# Visualisasi histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(sample_scores, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(stats["mean"], color='red', linestyle='--', label=f"Mean={stats['mean']:.1f}")
axes[0].set_title("Distribusi Cosine Similarity (Random Pairs)")
axes[0].set_xlabel("Similarity Score")
axes[0].set_ylabel("Frekuensi")
axes[0].legend()

# Disability distribution
disability_counts = pd.Series([m["disability_type"] for m in metadata.values()]).value_counts()
disability_counts.plot(kind="barh", ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title("Distribusi Job per Jenis Disabilitas")
axes[1].set_xlabel("Jumlah Jobs")

plt.tight_layout()
plt.savefig("data/processed/evaluation_plots.png", dpi=120, bbox_inches='tight')
plt.show()
print("💾 Plot → data/processed/evaluation_plots.png")

## 5. Simpan Laporan Evaluasi

In [ ]:
report = {
    "clustering"        : {"silhouette_score": sil_score, "n_classes": len(le.classes_)},
    "score_distribution": stats,
    "overall_grade"     : grade,
}

report_path = "data/processed/evaluation_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"💾 Laporan evaluasi → {report_path}")

## 6. Panduan Fine-Tuning (Contrastive Learning)

In [ ]:
# Kapan perlu fine-tuning?
print("Fine-tuning direkomendasikan jika:")
print("  - Silhouette score < 0.3")
print("  - Banyak rekomendasi tidak relevan saat manual review")
print("  - Model terlalu generik untuk konteks disabilitas Indonesia")

In [ ]:
import random
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader

def generate_training_pairs(jobs_df, kandidat_df, n_pairs=1000):
    """Buat pasangan training (positive & negative) dari dataset."""
    pairs = []
    for _, kandidat in kandidat_df.iterrows():
        k_disability = kandidat["disability_type"]
        k_query = kandidat["query_text"]

        positive_jobs = jobs_df[
            jobs_df["Jenis Disabilitas"].str.lower().str.contains(
                k_disability.lower().split("(")[0].strip(), na=False
            )
        ]
        negative_jobs = jobs_df[
            ~jobs_df["Jenis Disabilitas"].str.lower().str.contains(
                k_disability.lower().split("(")[0].strip(), na=False
            )
        ]
        if positive_jobs.empty or negative_jobs.empty:
            continue
        for _ in range(3):
            pos = positive_jobs.sample(1).iloc[0]
            neg = negative_jobs.sample(1).iloc[0]
            pairs.append(InputExample(texts=[k_query, pos["document_text"]], label=1.0))
            pairs.append(InputExample(texts=[k_query, neg["document_text"]], label=0.0))
    random.shuffle(pairs)
    return pairs[:n_pairs]


def fine_tune_model(model_name, training_pairs, output_dir="models/finetuned"):
    """Fine-tune model dengan CosineSimilarityLoss."""
    model = SentenceTransformer(model_name)
    train_dataloader = DataLoader(training_pairs, shuffle=True, batch_size=16)
    train_loss = losses.CosineSimilarityLoss(model)
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=3, warmup_steps=100,
        output_path=output_dir,
        show_progress_bar=True,
    )
    print(f"✅ Fine-tuned model disimpan → {output_dir}")
    return model


print("✅ Fungsi fine-tuning terdefinisi")
print("\n▶️  Cara menjalankan fine-tuning:")
print("   1. Load data yang sudah diproses")
print("   2. Panggil generate_training_pairs(jobs_df, kandidat_df, n_pairs=2000)")
print("   3. Panggil fine_tune_model(model_name, pairs, output_dir='models/finetuned')")
print("   4. Rebuild index (jalankan ulang 02_embedding.ipynb)")

In [ ]:
# === JALANKAN FINE-TUNING (uncomment jika siap) ===
# jobs_df = pd.read_csv("data/processed/jobs_processed.csv")
# kandidat_df = pd.read_csv("data/processed/kandidat_processed.csv")
#
# pairs = generate_training_pairs(jobs_df, kandidat_df, n_pairs=2000)
# print(f"Total training pairs: {len(pairs)}")
#
# model = fine_tune_model(
#     model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     training_pairs=pairs,
#     output_dir="models/finetuned"
# )
print("⚠️  Uncomment cell di atas untuk menjalankan fine-tuning")